# Week 4 – Seoul Bike Sharing: Model–Observation Synchronization

**Goal:** Turn the Week 3 simulation into a simple Digital Twin prototype (replay mode).

Each hour the twin:
1. **Synchronizes** its demand state with the new observation: `D_updated = D_sim + K · (D_obs − D_sim)`
2. **Predicts** the next hour: `D(t+1) = D_updated(t) + ΔD(t)`
3. **Compares** the prediction with the next recorded rental count.

`ΔD(t)` is estimated from time of day, weather and historical demand, using the Week 3 model `S(t)`:
`ΔD(t) = S(t+1) − S(t)`.

Historical CSV data is replayed hour by hour. This is a **Digital Twin prototype in replay mode**, not a live operational twin.

## 1. Import libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load data and rebuild the Week 3 model

Same data checks, train/test split and model as in `Week-3/week3_seoul_bike.ipynb`.
All averages and weather effects use **training data only**.

In [ ]:
csv = Path('../SeoulBikeData.csv')
if not csv.exists():
    csv = Path('SeoulBikeData.csv')
df = pd.read_csv(csv, encoding='latin1')
df['Timestamp'] = pd.to_datetime(df['Date'], dayfirst=True) + pd.to_timedelta(df['Hour'], unit='h')

weather = ['Temperature(°C)', 'Humidity(%)', 'Rainfall(mm)', 'Wind speed (m/s)']
cols = ['Timestamp', 'Hour', 'Functioning Day', 'Rented Bike Count'] + weather
df = df[cols].dropna().drop_duplicates('Timestamp').sort_values('Timestamp').copy()
df['Month'] = df['Timestamp'].dt.month
df['Day'] = df['Timestamp'].dt.date

# Split whole days within each month (80% train, 20% test)
train_parts, test_parts = [], []
for month, group in df.groupby('Month'):
    days = sorted(group['Day'].unique())
    cutoff = int(len(days) * 0.8)
    train_parts.append(group[group['Day'].isin(days[:cutoff])])
    test_parts.append(group[group['Day'].isin(days[cutoff:])])
train = pd.concat(train_parts).sort_values('Timestamp').copy()
test = pd.concat(test_parts).sort_values('Timestamp').copy()
print('Training rows:', len(train), '| Test rows:', len(test))

In [ ]:
# Month-hour baseline and monthly weather means (training data only)
active = train[train['Functioning Day'] == 'Yes'].copy()
monthly_weather = train.groupby('Month')[weather].mean()
rental_average = active.groupby(['Month', 'Hour'])['Rented Bike Count'].mean()

# Weather effects: relative rentals below / at-or-above the monthly weather mean
active['Baseline'] = [rental_average.loc[m, h] for m, h in zip(active['Month'], active['Hour'])]
active = active[active['Baseline'] > 0].copy()
active['Relative Rentals'] = active['Rented Bike Count'] / active['Baseline']

weather_effects = {}
for column in weather:
    above = active[column] >= active['Month'].map(monthly_weather[column])
    averages = active.groupby(['Month', above])['Relative Rentals'].mean()
    for month in monthly_weather.index:
        for is_above in [False, True]:
            weather_effects[(month, column, is_above)] = averages.get((month, is_above), 1.0)

In [ ]:
def simulate(row):
    """Week 3 model S(t): month-hour baseline x weather factors.
    Uses only known predictors (month, hour, weather, service status).
    Returns a float - no rounding inside the model."""
    if row['Functioning Day'] == 'No':
        return 0.0
    month, hour = row['Month'], row['Hour']
    estimate = rental_average.loc[month, hour]
    for column in weather:
        is_above = row[column] >= monthly_weather.loc[month, column]
        estimate *= weather_effects[(month, column, is_above)]
    return float(estimate)

train['S'] = train.apply(simulate, axis=1)
test['S'] = test.apply(simulate, axis=1)

## 3. Synchronization and one-hour-ahead prediction

For each hour `t` (in time order):

| Step | Formula |
|---|---|
| Simulated state | `D_sim(t)` = the twin's prediction for hour t (or `S(t)` at the start) |
| Synchronize | `D_updated(t) = D_sim(t) + K · (D_obs(t) − D_sim(t))` |
| Demand change | `ΔD(t) = S(t+1) − S(t)` |
| Predict | `D(t+1) = D_updated(t) + ΔD(t)` |
| Compare | `|report(t+1) − D_obs(t+1)|` |

**Gain K** (0–1) controls how much the twin trusts the observation:
`K = 0` → no synchronization (Week 3 model), `K = 1` → state fully replaced by the observation.

**Restart rule:** the test set has gaps between days of different months. When the next row is not exactly one hour later, the state is reset to `S(t)`.

**Rounding choice:** the internal state is kept as a float and may temporarily be negative. Rounding and clamping (`max(0, round(D))`) are applied **only to the reported prediction**, so rounding errors do not accumulate in the state.

**Service shutdown:** when the next hour is a non-functioning hour, completed rentals are zero by definition. These hours are excluded from the error, because predicting them is trivial.

In [ ]:
def run_twin(data, K):
    """Replay the observations hour by hour and return one-hour-ahead predictions."""
    rows = data.to_dict('records')
    results = []
    D = None  # twin demand state (None = not initialised)

    for now, nxt in zip(rows[:-1], rows[1:]):
        # Reset the state if the next row is not the next hour
        if nxt['Timestamp'] - now['Timestamp'] != pd.Timedelta(hours=1):
            D = None
            continue

        D_sim = now['S'] if D is None else D
        D_obs = now['Rented Bike Count']

        D_updated = D_sim + K * (D_obs - D_sim)   # synchronize
        delta_D = nxt['S'] - now['S']             # change from known predictors
        D = D_updated + delta_D                   # predict next hour

        results.append({
            'Timestamp': nxt['Timestamp'],
            'Measured': nxt['Rented Bike Count'],
            'Week 3 simulation': max(0, round(nxt['S'])),
            'Twin prediction': max(0, round(D)),    # clamp/round only for reporting
            'Operating': nxt['Functioning Day'] == 'Yes',
        })
    return pd.DataFrame(results)


def mae(results, column):
    op = results[results['Operating']]
    return (op[column] - op['Measured']).abs().mean()

## 4. Gain K

We use a **fixed gain K = 0.5**: the twin moves halfway between its own simulation and the new observation.

Why 0.5:
- It gives equal weight to the model and the observation.
- One noisy or wrong measurement only shifts the state by half of its error.
- It is a simple, explainable starting point. K can be tuned in a later week.

The table below is only a **sensitivity check on training data** (test data is not used). It shows how the error changes with K.

In [ ]:
K = 0.5

k_values = [0.0, 0.25, 0.5, 0.75, 1.0]
k_errors = pd.Series({k: mae(run_twin(train, k), 'Twin prediction') for k in k_values})

print('Sensitivity check - training MAE (operating hours) for different K:')
print(k_errors.round(1))
print('Used K:', K)

## 5. Evaluate on the test data

Compare three predictors on the same one-hour-ahead test hours:
- **Week 3 simulation** – no synchronization.
- **Twin, K = 0** – the dynamic model without observations (should match the Week 3 simulation).
- **Twin, K = 0.5** – the synchronized Digital Twin.

In [ ]:
twin = run_twin(test, K)
no_sync = run_twin(test, 0.0)

summary = pd.Series({
    'Week 3 simulation': mae(twin, 'Week 3 simulation'),
    'Twin, K = 0': mae(no_sync, 'Twin prediction'),
    f'Twin, K = {K}': mae(twin, 'Twin prediction'),
})
print('Test MAE, operating hours only (bicycles/hour):')
print(summary.round(1))
print('Evaluated hours:', int(twin['Operating'].sum()))

improvement = 1 - summary.iloc[2] / summary.iloc[0]
print(f'Error reduction from synchronization: {improvement:.0%}')

In [ ]:
# One test week from the first test month
first_month = twin['Timestamp'].dt.month.iloc[0]
week = twin[twin['Timestamp'].dt.month == first_month].head(168)

plt.figure(figsize=(12, 5))
plt.plot(week['Timestamp'], week['Measured'], label='Measured', linewidth=2)
plt.plot(week['Timestamp'], week['Week 3 simulation'], label='Week 3 simulation', alpha=0.7)
plt.plot(week['Timestamp'], week['Twin prediction'], label=f'Twin prediction (K = {K})')
plt.title('One-hour-ahead prediction: simulation vs synchronized twin')
plt.xlabel('Time')
plt.ylabel('Bicycles/hour')
plt.xticks(rotation=30)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Discussion

- Synchronization uses the latest observation to correct the demand state, so the twin follows unusual hours (e.g. sudden rain, events) much better than the static Week 3 simulation.
- With K = 0.5 the twin trusts the model and the observation equally. A higher K would follow the observations more closely (lower error in the sensitivity check), but a single noisy or wrong measurement would affect the next prediction more strongly. K = 0.5 is a compromise between accuracy and robustness.
- The prediction is only **one hour ahead**. For longer horizons the correction would fade and the model would rely more on `S(t)`.
- Weather inputs are treated as known for the next hour (perfect forecast). A real twin would use weather forecasts, which adds error.
- Completed rentals are a proxy for demand. During a shutdown, zero rentals do not mean zero demand.
- K is a single constant. It could be tuned per hour of day, or estimated with a Kalman filter in a later week.